# Vertical & Tier Segmentation Audit

Evaluates the global XGBoost (and other trained models) on per-segment subsets of the
validation set, broken out independently by **vertical** and **tier**.

This reveals whether the global model has blind spots across content domains or channel
sizes, and informs whether segment-specific models are worth investment.

**Flag threshold**: any segment where `roc_auc` drops > 0.03 below the global model AUC
(XGBoost global ≈ 0.909) is a candidate for a segment-specific model.

---
**Pipeline order**: run this audit *after* Validator (Stage 8) and *before* any holdout
evaluation. No GCS writes are performed in this notebook.

In [1]:
import sys
import os

# Adds src/capstone to path — mirrors the convention in other pipeline notebooks.
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory
from evaluation.segment_auditor import SegmentAuditor

---
## Config

Load-only run: reads the current data and model snapshots from GCS, trains models
on the current split, then validates. No snapshots are written.

To audit a specific saved version, swap `VersionConfig.load()` for a pinned config,
and use `stages.loader.run(run)` with `ModelLoader` instead of `ModelTrainer`.

In [2]:
config = VersionConfig.load(use_synthetic=False).build()
run = PipelineRun(config)
stages = PipelineFactory.retrain_existing_data(config)

print('Scenario :', stages.scenario)
print('Data     :', config.raw_version)
print('Model    :', config.model_version)

VersionConfig loaded:
  data:          v3.4 (raw_suffix='real')
  baselines:     v4.0
  model:         v5.3
  hyperparams:   v1.1
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['none (dry run)']
  data              : v3.4 (unchanged)
  baselines         : v4.0 (unchanged)
  model             : v5.3 (unchanged)
  hyperparams       : v1.1 (unchanged)
  raw_version       : v3.4_real
  next_final_version: v3.4_100real
  model_version     : v5.3
  baselines_version : v4.0
  hyperparam_version: v1.1
  use_synthetic     : False
Scenario : retrain_existing_data
Data     : v3.4_real
Model    : v5.3


---
## Stage 1 — DataLoader

In [3]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.4_real': 60696 rows from 2026-04-29
  Polls: {'upload': 21398, '24h': 20906, '7d': 18392}
Loaded baselines 'v4.0': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=[]))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  df_gen         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  X_gen          None
  y_train        None
  y_test         None
  y_val          None
  y_gen          None
  models         empty dict
  results        empty dict


---
## Stage 2 — DataPreprocessor

In [4]:
stages.preprocessor.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 18334 (dropped 3111 incomplete)
  Pivoted shape: (18403, 34)

[2/3] Joining baseline medians...
  Baseline join: 18403/18403 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 18403 rows × 40 columns

Clean dataset: 18403 rows × 40 columns
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=[]))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shap

---
## Stage 3 — FeatureEngineer

In [5]:
stages.engineer.run(run)
run.summary()

  all: dropped 414 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/10] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/10] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/10] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/10] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/10] Computing categorical features...
  Title categories:
title_category
neutral        11344
all_caps        2048
exclamation     1955
question        1572
listicle         562
how_to           230
clickbait        204
emoji_heavy       74
  Description categories:
desc_category
link_heavy        4729
minimal           4509
has_links         3837
has_hashtags      2697
neutral           1395
has_timestamps     774
long_form       

---
## Stage 4 — DataSplitter

In [6]:
stages.splitter.run(run)
run.summary()

DataSplitter — loaded holdout (5,193 val rows):
  df_val:    5,193 rows (28.9%)
  df_train: 10,236 rows (56.9%)
  df_test:   2,560 rows (14.2%)
DataSplitter — no generalization-vertical rows (Music/Sports) found.
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=[]))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 89)
  df_train       populated  DataFrame shape=(10236, 89)
  df_test        populated  DataFrame shape=(2560, 89)
  df_val         populated  DataFrame shape=(5193, 89)
  df_gen         populated  DataFrame shape=(0, 89)
  X_train        populated  DataFrame shape=(10236, 55)
  X_test         populated  DataFrame shape=(2560, 55)
  X_val          populated  DataFrame shape=(5193,

---
## Stage 5 — Scaler

In [7]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=[]))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 89)
  df_train       populated  DataFrame shape=(10236, 89)
  df_test        populated  DataFrame shape=(2560, 89)
  df_val         populated  DataFrame shape=(5193, 89)
  df_gen         populated  DataFrame shape=(0, 89)
  X_train        populated  DataFrame shape=(10236, 55)
  X_test         populated  DataFrame shape=(2560, 55)
  X_val          populated  DataFrame shape=(5193, 55)
  X_val_unscaled  populated  DataFrame shape=(5193, 55)
  X_gen          populated  DataFrame shape=(0, 55)
  y_train        populated  Series length=10236
  y_test         populated  Series length=2560
  y_v

---
## Stage 6 — ModelTrainer

> **Note**: `run.X_val` produced by the Scaler uses the same `StandardScaler` fitted
> on this run's training data. Since models are trained in the same pipeline run, `run.X_val`
> is the correct input for `SegmentAuditor`. If you load historical models via `ModelLoader`,
> pass `run.X_val_unscaled` and apply each model's own scaler (as `ValidatorLogic` does).

In [8]:
stages.trainer.run(run)
run.summary()

Loaded hyperparams 'v1.1' (saved 2026-04-29)
  Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier']
  Search: {'strategy': 'random', 'n_iter': 100, 'cv': 5, 'scoring': 'roc_auc'}
Loaded hyperparams from snapshot 'v1.1'.
  Note: injected l1_ratio=0.5 for elasticnet LR (missing from snapshot).
Training LogisticRegression (L1)...
Training RandomForestClassifier...
Training XGBClassifier...
Training VotingClassifier ensemble (RF + XGB, weights=[1, 2])...

=== ModelTrainer — test-set results ===
  lr_l1         AUC=0.7719  acc=0.7117  F1↑=0.7446
  rf            AUC=0.8645  acc=0.7805  F1↑=0.8079
  xgb           AUC=0.9085  acc=0.8262  F1↑=0.8446
  ensemble      AUC=0.9049  acc=0.8203  F1↑=0.8398
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=[]))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  Data

---
## Stage 7 — Validator (Global Baseline)

In [9]:
stages.validator.run(run)

metric_cols = ['roc_auc', 'accuracy', 'f1_above', 'precision_above', 'recall_above', 'recall_below']
df_global = (
    pd.DataFrame(run.results).T[metric_cols]
    .astype(float)
    .round(4)
)
df_global.index.name = 'model'
print('\n=== Global validation-set results ===')
df_global


=== Validator — validation-set results ===
  lr_l1           AUC=0.7681  acc=0.7004  F1↑=0.7330
  rf              AUC=0.8718  acc=0.7928  F1↑=0.8204
  xgb             AUC=0.9090  acc=0.8302  F1↑=0.8487
  ensemble        AUC=0.9058  acc=0.8305  F1↑=0.8495

=== Global validation-set results ===


,roc_auc,accuracy,f1_above,precision_above,recall_above,recall_below
model,,,,,,
lr_l1,0.7681,0.7004,0.7330,0.7255,0.7406,0.6501
rf,0.8718,0.7928,0.8204,0.7910,0.8519,0.7189
xgb,0.9090,0.8302,0.8487,0.8400,0.8575,0.7960
ensemble,0.9058,0.8305,0.8495,0.8383,0.8610,0.7926


---
## Segment Audit

Check alignment invariants, then run per-segment evaluation across all models.

In [10]:
# Verify df_val has the segment label columns we need.
print('df_val columns with segment labels:')
print('  vertical unique:', sorted(run.df_val['vertical'].unique()))
print('  tier unique    :', sorted(run.df_val['tier'].unique()))
print('  df_val rows    :', len(run.df_val))
print('  X_val rows     :', run.X_val.shape[0])

assert len(run.df_val) == run.X_val.shape[0], (
    f'Row count mismatch: df_val={len(run.df_val)}, X_val={run.X_val.shape[0]}'
)

df_val columns with segment labels:
  vertical unique: ['Education', 'Lifestyle', 'Tech']
  tier unique    : ['L', 'M', 'S']
  df_val rows    : 5193
  X_val rows     : 5193


In [11]:
# Using 0.5 for an apples-to-apples comparison with global validation metrics.
# Swap in the optimized threshold (e.g. 0.58) only if you want to audit
# segment performance under the same decision boundary used in production.
THRESHOLD = 0.5

auditor = SegmentAuditor(
    models=run.models,        # dict of {model_name: run.models entry}
    X_val=run.X_val,          # globally scaled validation feature matrix
    y_val=run.y_val,          # binary validation labels (above_baseline)
    df_val=run.df_val,        # pre-FE split — provides 'vertical' and 'tier' labels
    threshold=THRESHOLD,
)

segment_results = auditor.audit()
print(f'Audit complete: {len(segment_results)} segment × model rows')

Audit complete: 24 segment × model rows


---
## Summary Table

In [12]:
auditor.print_summary(segment_results)


  Segment audit — VERTICAL
vertical      ensemble  auc    lr_l1  auc    rf  auc    xgb  auc    ensemble  acc    lr_l1  acc    rf  acc    xgb  acc
----------  ---------------  ------------  ---------  ----------  ---------------  ------------  ---------  ----------
Education            0.9000        0.7745     0.8746      0.9017           0.8276        0.7100     0.8077      0.8245
Lifestyle            0.9267        0.7701     0.8910      0.9315           0.8553        0.7122     0.8074      0.8596
Tech                 0.8873        0.7588     0.8465      0.8906           0.8072        0.6791     0.7637      0.8043
ALL                  0.9058        0.7681     0.8718      0.9090           0.8305        0.7004     0.7928      0.8302

  Blind spots (roc_auc > 0.03 below global):
    None — model performs uniformly across all segments.

  Segment audit — TIER
tier      ensemble  auc    lr_l1  auc    rf  auc    xgb  auc    ensemble  acc    lr_l1  acc    rf  acc    xgb  acc
------  --------

---
## Full Results Table

In [13]:
display_cols = [
    'model', 'segment_type', 'segment_value', 'n_samples',
    'pct_positive', 'roc_auc', 'accuracy', 'f1_above', 'recall_below',
]
segment_results[display_cols].style \
    .format({
        'pct_positive': '{:.2%}',
        'roc_auc':      '{:.4f}',
        'accuracy':     '{:.4f}',
        'f1_above':     '{:.4f}',
        'recall_below': '{:.4f}',
    }) \
    .background_gradient(subset=['roc_auc'], cmap='RdYlGn', vmin=0.7, vmax=1.0)

,model,segment_type,segment_value,n_samples,pct_positive,roc_auc,accuracy,f1_above,recall_below
0,ensemble,tier,L,1785,53.67%,0.9400,0.8639,0.8732,0.8525
1,ensemble,tier,M,1865,58.02%,0.9019,0.8322,0.8578,0.7765
2,ensemble,tier,S,1543,54.70%,0.8647,0.7900,0.8125,0.7396
3,lr_l1,tier,L,1785,53.67%,0.7846,0.7031,0.7329,0.6385
4,lr_l1,tier,M,1865,58.02%,0.7573,0.6949,0.7360,0.6424
5,lr_l1,tier,S,1543,54.70%,0.7679,0.7038,0.7294,0.6724
6,rf,tier,L,1785,53.67%,0.9119,0.8370,0.8499,0.8102
7,rf,tier,M,1865,58.02%,0.8635,0.7834,0.8217,0.6769
8,rf,tier,S,1543,54.70%,0.8314,0.7531,0.7866,0.6581
9,xgb,tier,L,1785,53.67%,0.9421,0.8661,0.8752,0.8561


---
## Blind-Spot Analysis

Flag any segment where XGBoost `roc_auc` drops more than **0.03 below** the global
XGBoost AUC (~0.909). Those are candidates for segment-specific model investment.

In [14]:
XGB_GLOBAL_AUC = run.results.get('xgb', {}).get('roc_auc', None)
BLIND_SPOT_DROP = 0.03

if XGB_GLOBAL_AUC is not None:
    xgb_segs = segment_results[segment_results['model'] == 'xgb'].copy()
    xgb_segs['auc_drop'] = XGB_GLOBAL_AUC - xgb_segs['roc_auc']
    blind_spots = xgb_segs[xgb_segs['auc_drop'] > BLIND_SPOT_DROP].sort_values('auc_drop', ascending=False)

    print(f'XGBoost global AUC : {XGB_GLOBAL_AUC:.4f}')
    print(f'Blind-spot threshold: > {BLIND_SPOT_DROP} drop')
    print()

    if blind_spots.empty:
        print('No blind spots detected — XGBoost performs uniformly across all segments.')
    else:
        print(f'{len(blind_spots)} blind spot(s) detected:')
        display(blind_spots[['segment_type', 'segment_value', 'n_samples', 'roc_auc', 'auc_drop']].reset_index(drop=True))
else:
    print('XGBoost results not found in run.results — did Validator run?')

XGBoost global AUC : 0.9090
Blind-spot threshold: > 0.03 drop

1 blind spot(s) detected:


,segment_type,segment_value,n_samples,roc_auc,auc_drop
0,tier,S,1543,0.867612,0.041388


---
## Persist Results (Optional)

In [15]:
import os

output_dir = os.path.abspath('../../../data/segment_audits')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, f'segment_audit_{config.model_version}.csv')
segment_results.to_csv(output_path, index=False)
print(f'Saved → {output_path}')

Saved → /Users/jelanigould-bailey/Desktop/uc_berkeley_capstone_repo/repo/data/segment_audits/segment_audit_v5.3.csv
